# Michaelis-Menten
***
## Setup the Environment
***

In [2]:
%load_ext autoreload
%autoreload 2

In [17]:
import os
import sys
from copy import deepcopy
sys.path.insert(1, os.path.abspath(os.path.join(os.getcwd(), '../')))

In [18]:
import gillespy3d_pp as gillespy2

In [19]:
gillespy2.__file__

'/Users/anisgolriz/Desktop/GillesPy3D/simulation_lib/gillespy3d_pp/__init__.py'

In [20]:
def create_michaelis_menten(parameter_values=None):
    # Initialize Model
    model = gillespy2.Model(name="Michaelis_Menten")

    # Define Variables (GillesPy2.Species)
    A = gillespy2.Species(name='Substrate', initial_value=301)
    B = gillespy2.Species(name='Enzyme', initial_value=120)
    C = gillespy2.Species(name='Enzyme_Substrate_Complex', 
                          initial_value=0)
    D = gillespy2.Species(name='Product', initial_value=0)
    
    # Add Variables to Model
    model.add_species([A, B, C, D])
    

    # Define Parameters
    rate1 = gillespy2.Parameter(name='rate1', expression=0.0017)
    rate2 = gillespy2.Parameter(name='rate2', expression=0.5)
    rate3 = gillespy2.Parameter(name='rate3', expression=0.1)
    
    # Add Parameters to Model
    model.add_parameter([rate1, rate2, rate3])
    
    # Define Reactions
    r1 = gillespy2.Reaction(
        name="r1", reactants={'Substrate': 1, 'Enzyme': 1}, 
        products={'Enzyme_Substrate_Complex': 1}, rate='rate1'
    )
    r2 = gillespy2.Reaction(
        name="r2", reactants={'Enzyme_Substrate_Complex': 1}, 
        products={'Substrate': 1, 'Enzyme': 1}, rate='rate2'
    )
    r3 = gillespy2.Reaction(
        name="r3", reactants={'Enzyme_Substrate_Complex': 1}, 
        products={'Enzyme': 1, 'Product': 1}, rate='rate3'
    )
    
    # Add Reactions to Model
    model.add_reaction([r1, r2, r3])
    
    # Define Timespan
    tspan = gillespy2.TimeSpan.linspace(t=100, num_points=101)
    
    # Set Model Timespan
    model.timespan(tspan)
    return model

### Instantiate the Model

In [22]:
model = create_michaelis_menten()

AttributeError: 'Model' object has no attribute 'deepcopy'

In [8]:
dir(gillespy2)

['Model',
 'NumPySSASolver',
 'Parameter',
 'Reaction',
 'Result',
 'Simulation',
 'Species',
 'TimeSpan',
 '__author__',
 '__builtins__',
 '__cached__',
 '__copyright__',
 '__description__',
 '__doc__',
 '__email__',
 '__file__',
 '__license__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__title__',
 '__url__',
 '__version__',
 'core',
 'error',
 'log',
 'logging',
 'model',
 'parameter',
 'reaction',
 'result',
 'simulation',
 'solvers',
 'species',
 'sys',
 'timespan',
 'utils',
 'version']

In [9]:
# The GillesPy2 way to run GillesPy3D
result = model.run()
result.plot()

AttributeError: 'Result' object has no attribute 'plot'

In [23]:
import gillespy3d_pp.utils.solverutils as nputils
import numpy as np
if model is None:
    raise NumPySSASolverError("A model is required to run the simulation.")
species, species_mappings, parameter_mappings, number_species = nputils.numpy_initialization(model)
        #second line possible dupe?
        #self.species = self.model.listOfSpecies
reactions = list(model.listOfReactions.keys())
number_reactions = len(reactions)
dependent_rxns = nputils.dependency_grapher(model, reactions)
is_instantiated = True
number_species = len(model.listOfSpecies)
species_changes = np.zeros((number_reactions,number_species))
propensity_functions = {}
volume = getattr(model, "volume", 1.0)
parameters = {'V': volume}
species_mappings  = model._sanitized_species_names()#solver utils
parameter_mappings = model._sanitized_parameter_names()
for i, r_name in enumerate(reactions):
    for j,(s_name, spec) in enumerate(species.items()):
        species_changes[i][j] = model.listOfReactions[r_name].products.get(model.listOfSpecies[s_name], 0) \
                                - model.listOfReactions[r_name].reactants.get(model.listOfSpecies[s_name], 0)

    propensity_functions[r_name] = [eval('lambda S:' + model.listOfReactions[r_name].
                                                sanitized_propensity_function(species_mappings, parameter_mappings),
                                                   parameters), i]

ValueError: not enough values to unpack (expected 4, got 3)

In [11]:
reactions

NameError: name 'reactions' is not defined

In [24]:
# the other way to run GillesPy3D
sim = gillespy2.Simulation(model, "SSA")
#sim = gillespy2.Simulation(model, NumPySSASolver())

num_traj = 5
for traj in range(num_traj):
    sim.reset() # reset the simulation after each run
    dt = .1;
    end_t = 10;
    while sim.get_time() < end_t:
        print('traj',traj,' t:',sim.get_time(),' Substrate:',sim.get_species('Substrate'))
        sim.run_until(sim.get_time()+dt);

dependent reac  {'r1': {'dependencies': []}, 'r2': {'dependencies': ['r1']}, 'r3': {'dependencies': ['r1']}}
dependent reac  {'r1': {'dependencies': ['r2']}, 'r2': {'dependencies': ['r1', 'r3']}, 'r3': {'dependencies': ['r1', 'r2']}}
dependent reac  {'r1': {'dependencies': ['r2', 'r3']}, 'r2': {'dependencies': ['r1', 'r3']}, 'r3': {'dependencies': ['r1', 'r2']}}
PARAM IN SOLVER: rate1 '0'
PARAM IN SOLVER: rate2 '0'
PARAM IN SOLVER: rate3 '0'
parameter_mappings is  {'rate1': 0.0, 'rate2': 0.0, 'rate3': 0.0, 'vol': 1.0}
sanitized_propensity before is  (((rate1*Substrate)*Enzyme)/vol)
sanitized_propensity is  ((({4}*{1})*{3})/{7})
FINAL LAMBDA: (((0.0*S[0])*S[1])/1.0)


RuntimeError: No active exception to reraise